In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("NASA TurboFan Engine.csv")
features = df.filter(like="sensor").columns
train = df[(df.unit <= 80) & (df.Fault_Label == 0)]
test = df[df.unit > 80]
scaler = StandardScaler().fit(train[features])
datasets = []
for frame in [train, test]:
    windows = []
    for _, engine in frame.groupby("unit"):
        values = scaler.transform(engine.sort_values("cycle")[features])
        windows.extend([values[i:i+20] for i in range(len(values)-19)])
    datasets.append(np.array(windows))
x_train, x_test = datasets

In [ ]:
model = Sequential([Input((20, 21)), LSTM(16), RepeatVector(20),
                    LSTM(16, return_sequences=True), Dense(21)])
model.compile(optimizer="adam", loss="mse")
model.fit(x_train, x_train, epochs=10, batch_size=128)

In [ ]:
error = np.mean((x_train - model.predict(x_train, verbose=0)) ** 2, axis=(1, 2))
threshold = np.percentile(error, 95)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=(1, 2))
anomalies = error > threshold
print("Anomalies:", anomalies.sum())